# Dataset ALLFLOWMETER_HIKARI2021

In [31]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
#import seaborn as sns
from scipy import stats
from sklearn.feature_selection import mutual_info_regression
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression, LogisticRegression

# Carregamento e limpeza do dataset
df = pd.read_csv("../../data/raw/ALLFLOWMETER_HIKARI2021.csv")
df.head()


,Unnamed: 0.1,Unnamed: 0,uid,originh,originp,responh,responp,flow_duration,fwd_pkts_tot,bwd_pkts_tot,...,idle.min,idle.max,idle.tot,idle.avg,idle.std,fwd_init_window_size,bwd_init_window_size,fwd_last_window_size,traffic_category,Label
0,0,0,Cg61Jch3vdz9DBptj,103.255.15.23,13316,128.199.242.104,443,2.207588,15,14,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0,29200,65160,0,Bruteforce-XML,1
1,1,1,CdRIlqLWdj35Y9vW9,103.255.15.23,13318,128.199.242.104,443,15.624266,15,14,...,1.534300e+07,1.534300e+07,1.534300e+07,1.534300e+07,0.0,29200,65160,0,Bruteforce-XML,1
2,2,2,CLzp9Khd0Y09Qkgrg,103.255.15.23,13320,128.199.242.104,443,12.203357,14,13,...,1.196814e+07,1.196814e+07,1.196814e+07,1.196814e+07,0.0,29200,65160,0,Bruteforce-XML,1
3,3,3,Cnf1YA4iLB4CSNWB88,103.255.15.23,13322,128.199.242.104,443,9.992448,14,13,...,9.759205e+06,9.759205e+06,9.759205e+06,9.759205e+06,0.0,29200,65160,0,Bruteforce-XML,1
4,4,4,C4ZKvv3fpO72EAOsJ6,103.255.15.23,13324,128.199.242.104,443,7.780611,14,14,...,7.545305e+06,7.545305e+06,7.545305e+06,7.545305e+06,0.0,29200,65160,0,Bruteforce-XML,1


inspect the dataset

In [32]:
df.shape
df.info()
df.columns.tolist()
df.isna().sum().sort_values(ascending=False).head(20)
df.duplicated().sum()

<class 'pandas.DataFrame'>
RangeIndex: 555278 entries, 0 to 555277
Data columns (total 88 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   Unnamed: 0.1              555278 non-null  int64  
 1   Unnamed: 0                555278 non-null  int64  
 2   uid                       555278 non-null  str    
 3   originh                   555278 non-null  str    
 4   originp                   555278 non-null  int64  
 5   responh                   555278 non-null  str    
 6   responp                   555278 non-null  int64  
 7   flow_duration             555278 non-null  float64
 8   fwd_pkts_tot              555278 non-null  int64  
 9   bwd_pkts_tot              555278 non-null  int64  
 10  fwd_data_pkts_tot         555278 non-null  int64  
 11  bwd_data_pkts_tot         555278 non-null  int64  
 12  fwd_pkts_per_sec          555278 non-null  float64
 13  bwd_pkts_per_sec          555278 non-null  float64
 14 

np.int64(0)

Find the label columns

In [33]:
for col in df.columns:
    print(col, df[col].nunique(), df[col].dropna().unique()[:10])

Unnamed: 0.1 555278 [0 1 2 3 4 5 6 7 8 9]
Unnamed: 0 350710 [0 1 2 3 4 5 6 7 8 9]
uid 555278 <StringArray>
[ 'Cg61Jch3vdz9DBptj',  'CdRIlqLWdj35Y9vW9',  'CLzp9Khd0Y09Qkgrg',
 'Cnf1YA4iLB4CSNWB88', 'C4ZKvv3fpO72EAOsJ6',  'CyC8D5X7IIG7U95l4',
 'CEXyM013OxRuUddrS2', 'CVFc4q26WLSGblwO2c', 'CCvZhO2f7ztLs9Hopc',
  'CIPZU1mfhrkqqix49']
Length: 10, dtype: str
originh 2899 <StringArray>
[            '103.255.15.23',             '103.255.15.27',
             '103.255.15.20',                   '0.0.0.0',
            '202.169.224.77',             '103.255.15.40',
            '103.255.15.206', 'fe80::c1a7:7791:969e:3c06',
             '103.255.15.21',             '103.255.15.65']
Length: 10, dtype: str
originp 62886 [13316 13318 13320 13322 13324 13326 13328 13330 13332 13334]
responh 7991 <StringArray>
[   '128.199.242.104',      '128.199.88.81',      '103.255.15.23',
     '103.255.15.255',    '255.255.255.255',            '8.8.8.8',
      '34.107.221.82', '2600:1901:0:38d7::',       '54.230.151.7

Check class balance

In [34]:
df["Label"].value_counts()
df["Label"].value_counts(normalize=True) * 100



Label
0    93.211328
1     6.788672
Name: proportion, dtype: float64

In [35]:
df["traffic_category"].value_counts()
df["traffic_category"].value_counts(normalize=True) * 100

traffic_category
Benign                 62.568839
Background             30.642489
Probing                 4.211944
Bruteforce              1.059649
Bruteforce-XML          0.926563
XMRIGCC CryptoMiner     0.590515
Name: proportion, dtype: float64

In [36]:
df.columns[df.columns.str.contains("category|label|traffic", case=False)]
pd.crosstab(df["traffic_category"], df["Label"])

Label,0,1
traffic_category,,
Background,170151,0
Benign,347431,0
Bruteforce,0,5884
Bruteforce-XML,0,5145
Probing,0,23388
XMRIGCC CryptoMiner,0,3279


quick dataset summary

In [37]:
summary = {
    "rows": df.shape[0],
    "columns": df.shape[1],
    "duplicates": int(df.duplicated().sum()),
    "missing_values_total": int(df.isna().sum().sum())
}

summary

{'rows': 555278, 'columns': 88, 'duplicates': 0, 'missing_values_total': 0}

**Create X and y**

In [38]:
target_col = "Label"

drop_cols = [
    "Unnamed: 0.1",
    "Unnamed: 0",
    "uid",
    "originh",
    "responh",
    "traffic_category",
    "Label"
]

X = df.drop(columns=drop_cols, errors="ignore")
y = df[target_col]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (555278, 81)
y shape: (555278,)


Cleaning values

In [39]:
X = X.select_dtypes(include=["number"])

print("Final X shape:", X.shape)
print("Columns used:", X.columns.tolist())

#Clean missing or infinite values
X = X.replace([np.inf, -np.inf], np.nan)
print("Missing values before cleaning:", X.isna().sum().sum())
X = X.fillna(X.median(numeric_only=True))
print("Missing values after cleaning:", X.isna().sum().sum())



Final X shape: (555278, 81)
Columns used: ['originp', 'responp', 'flow_duration', 'fwd_pkts_tot', 'bwd_pkts_tot', 'fwd_data_pkts_tot', 'bwd_data_pkts_tot', 'fwd_pkts_per_sec', 'bwd_pkts_per_sec', 'flow_pkts_per_sec', 'down_up_ratio', 'fwd_header_size_tot', 'fwd_header_size_min', 'fwd_header_size_max', 'bwd_header_size_tot', 'bwd_header_size_min', 'bwd_header_size_max', 'flow_FIN_flag_count', 'flow_SYN_flag_count', 'flow_RST_flag_count', 'fwd_PSH_flag_count', 'bwd_PSH_flag_count', 'flow_ACK_flag_count', 'fwd_URG_flag_count', 'bwd_URG_flag_count', 'flow_CWR_flag_count', 'flow_ECE_flag_count', 'fwd_pkts_payload.min', 'fwd_pkts_payload.max', 'fwd_pkts_payload.tot', 'fwd_pkts_payload.avg', 'fwd_pkts_payload.std', 'bwd_pkts_payload.min', 'bwd_pkts_payload.max', 'bwd_pkts_payload.tot', 'bwd_pkts_payload.avg', 'bwd_pkts_payload.std', 'flow_pkts_payload.min', 'flow_pkts_payload.max', 'flow_pkts_payload.tot', 'flow_pkts_payload.avg', 'flow_pkts_payload.std', 'fwd_iat.min', 'fwd_iat.max', 'fwd_ia

# Split and train (70/30%)

In [47]:
#Train/test split 30% test size, stratified by the target variable
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30, #test size of 30%
    random_state=42,
    stratify=y
)

print("Training:", X_train.shape)
print("Testing:", X_test.shape)

Training: (388694, 81)
Testing: (166584, 81)


# Scale the data for Logistic Regression

In [48]:
from sklearn.preprocessing import StandardScaler
#Some models need scaling. Logistic Regression performs better when features are on comparable scales.
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train Model 1 — Logistic Regression

In [49]:
from sklearn.linear_model import LogisticRegression

log_model = LogisticRegression(
    max_iter=1000,
    random_state=42,
    class_weight="balanced"
)

log_model.fit(X_train_scaled, y_train)

y_pred_log = log_model.predict(X_test_scaled)

# Train Model 2 — Random Forest

In [50]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

# Evaluate both models

In [51]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

def evaluate_model(model_name, y_true, y_pred):
    results = {
        "Model": model_name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1-score": f1_score(y_true, y_pred, zero_division=0)
    }
    
    print(f"\n=== {model_name} ===")
    print(classification_report(y_true, y_pred, zero_division=0))
    print("Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))
    
    return results

# Evaluation

In [52]:
results_log = evaluate_model("Logistic Regression", y_test, y_pred_log)
results_rf = evaluate_model("Random Forest", y_test, y_pred_rf)


=== Logistic Regression ===
              precision    recall  f1-score   support

           0       1.00      0.83      0.91    155275
           1       0.30      0.99      0.46     11309

    accuracy                           0.84    166584
   macro avg       0.65      0.91      0.68    166584
weighted avg       0.95      0.84      0.88    166584

Confusion Matrix:
[[129008  26267]
 [    92  11217]]

=== Random Forest ===
              precision    recall  f1-score   support

           0       0.95      0.91      0.93    155275
           1       0.24      0.38      0.29     11309

    accuracy                           0.88    166584
   macro avg       0.60      0.65      0.61    166584
weighted avg       0.90      0.88      0.89    166584

Confusion Matrix:
[[141689  13586]
 [  7028   4281]]
